# 06 · Gegevens opvragen via een API

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

De diergewichtscomputer en de voedercomputer sturen hun metingen via MQTT naar de server. In notebook 05 heb je beide meetstromen in één database verzameld. Vanuit het kantoor wil je die gegevens nu opvragen om ze later samen te onderzoeken. Daarvoor breid je de bestaande webdienst uit met een API die meetgegevens kan lezen.

Je vraagt eerst enkele records op, kiest daarna zelf een selectie en vult ten slotte een aparte analysedatabase. Je controleert of deze kopie dezelfde meetinhoud bevat als de serverdatabase.

**Nodig:** dezelfde Codespace en eigen branch, je eigen Droplet uit notebook 03 en een geslaagde volledige ontvangst uit notebook 05. Houd je **`SSH beheer-wachtwoord`** en **`API-Droplet wachtwoord`** bij de hand. Je gebruikt beide bronnen, ongeacht je eigen rol.

Voer de codecellen in volgorde uit. Volg bij de serverstappen eerst de instructies in de SSH-terminal voordat je verdergaat met de volgende controle.

</div>

## 1. De werkplek en ontvangen gegevens controleren

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 06)

In deze oefening gebruik je Codespaces ook als de computer van het kantoor. Die computer vraagt gegevens op en bewaart een eigen analysebestand. Je hoeft hiervoor geen tweede Codespace te openen. De functies zijn gescheiden door verschillende bestanden en programma's.

De volgende cel haalt je rol, serveradres en certificaat uit de vorige oefeningen. Ze controleert ook of het inleverbestand uit notebook 05 bij jouw branch hoort. Je maakt de server en de accounts niet opnieuw aan.

</div>

In [1]:
_DOCENT_BRANCH_OVERRIDE = "student/proefdocent"

In [2]:
from pathlib import Path
from datetime import datetime, timezone
from getpass import getpass
from contextlib import closing
import ipaddress
import json
import re
import sqlite3
import subprocess
import sys
import pandas as pd

r = subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True)
if r.returncode:
    raise RuntimeError("Open notebook 06 vanuit de practicumrepository in Codespaces.")
hoofdmap = Path(r.stdout.strip())
branch = subprocess.check_output(["git", "branch", "--show-current"], cwd=hoofdmap, text=True).strip()
if not re.fullmatch(r"student/[a-z0-9][a-z0-9_-]*", branch) and "_DOCENT_BRANCH_OVERRIDE" in globals():
    branch = _DOCENT_BRANCH_OVERRIDE
instellingen = json.loads((hoofdmap / "gegevens/inlevering/01_werkplekcontrole.json").read_text(encoding="utf-8"))
deelnemer, rol = instellingen["deelnemerscode"], instellingen["rol"]
if branch != "student/" + deelnemer or instellingen.get("branch") != branch or rol not in ("gewichten", "voederopname"):
    raise ValueError("Controleer je eigen branch en instellingen uit notebook 01.")
werkmap = hoofdmap / "gegevens/werk"
server = json.loads((werkmap / "server.json").read_text(encoding="utf-8"))
if server.get("deelnemerscode") != deelnemer:
    raise ValueError("De serverinstellingen horen bij een andere deelnemer.")
serveradres = str(ipaddress.IPv4Address(server["ipv4"]))
ca = werkmap / "ca.crt"
if not ca.is_file():
    raise FileNotFoundError("Haal je openbare ca.crt opnieuw op volgens notebook 03.")
ontvangst05 = json.loads((hoofdmap / "gegevens/inlevering/05_ontvangstcontrole.json").read_text(encoding="utf-8"))
if (ontvangst05.get("deelnemerscode") != deelnemer or ontvangst05.get("branch") != branch
    or ontvangst05.get("volledigheid") != "geslaagd"
    or set(ontvangst05.get("bronnen", {})) != {"asserva", "schauer"}):
    raise ValueError("Rond de inhoudscontrole en inlevering van notebook 05 af.")
sys.path[:0] = [str(hoofdmap / "programmatuur"), str(hoofdmap / "server")]
from api_ophalen import ApiClient, werk_analyse_bij
print("Deelnemer:", deelnemer, "| Rol:", rol)
print("Beide bronnen gecontroleerd in notebook 05:", {b: v["aantal"] for b, v in ontvangst05["bronnen"].items()})

Deelnemer: proefdocent | Rol: gewichten
Beide bronnen gecontroleerd in notebook 05: {'asserva': 80028, 'schauer': 146649}


| Bestand of programma | Computer | Functie |
|---|---|---|
| `asserva.db` of `schauer.db` | Toestelcomputer in Codespaces | Brongegevens publiceren |
| Mosquitto en `iot-ontvanger` | Eigen Droplet | Berichten verdelen en opslaan |
| `verzamelde_metingen.db` | Eigen Droplet | Ontvangen gegevens van beide toestellen bewaren |
| FastAPI achter Nginx | Eigen Droplet | Gerichte HTTPS-aanvragen beantwoorden |
| `analyse_06.db` | Kantoorcomputer in Codespaces | Opgevraagde gegevens voor berekeningen bewaren |

![Gegevens van toestelcomputers via MQTT naar de server en via HTTPS naar de kantoorcomputer](../../afbeeldingen/api_gegevensstroom.svg)

*Figuur 1. De API leest de verzameldatabase. De kantoorcomputer ontvangt een antwoord op een aanvraag.*

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Vraag 1.** Waarom heeft de kantoorcomputer nog een API nodig als de metingen al via MQTT zijn verstuurd?

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Jouw antwoord (Vraag 1):**

*Voeg hier je antwoord in. Tip: vergelijk het ontvangen van nieuwe berichten met het later opvragen van opgeslagen gegevens.*

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Antwoord 1**

MQTT levert berichten aan actieve ontvangers. De ontvanger heeft die metingen op de server bewaard. Via de API kan de kantoorcomputer op een later moment een selectie uit die opgeslagen gegevens opvragen, zonder alle eerdere MQTT-berichten zelf te hebben gevolgd.

</div>

## 2. Wat gebeurt er bij een API-aanvraag?

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Een **API** beschrijft hoe programma's gegevens of functies kunnen gebruiken. Hier gebruik je een web-API: Python verstuurt een aanvraag naar een webadres en ontvangt een antwoord. **HTTP** legt vast hoe zo'n aanvraag en antwoord zijn opgebouwd. **HTTPS** voegt versleuteling en controle van de serveridentiteit toe.

Met **GET** vraag je gegevens op. De **URL** geeft het serveradres en het gevraagde pad aan. Met **parameters** verfijn je de selectie. Het antwoord bestaat uit een **statuscode**, informatie over het antwoord en een inhoud, hier in JSON. Een API biedt alleen de selecties die erin geprogrammeerd zijn.

In het voorbeeld hieronder vraagt de kantoorcomputer maximaal vijf gewichtsrecords van hok 1 op. Het adres is een voorbeeld, geen server waarmee je verbinding moet maken.

</div>

```text
GET https://<jouw-serveradres>/metingen?meettype=gewichten&hok=1&limiet=5
```

| Onderdeel | Betekenis |
|---|---|
| `https://` | Versleutelde webverbinding |
| `<jouw-serveradres>` | Je eigen Droplet |
| `/metingen` | Aanspreekpunt voor meetrecords |
| `?meettype=gewichten` | Begin van de parameters, kies het meettype |
| `&hok=1` | Beperk de selectie tot hokcode `1` |
| `&limiet=5` | Vraag maximaal vijf records in één antwoord |
| HTTP `200` | De aanvraag is gelukt, ook wanneer de selectie leeg is |

<div style="background-color:#e8f5e9;color:#1b5e20;border-left:5px solid #2e7d32;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;white-space:normal;"><strong>Uitleg</strong><br>FastAPI verwerkt de aanvraag op de server. Uvicorn voert dat Python-programma uit. Nginx verzorgt de HTTPS-verbinding en geeft de aanvraag intern door. Deze onderdelen zijn al geïnstalleerd in notebook 03. Het notebook is nu de client.</div>

## 3. De meetgegevens-API op je server installeren

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** eerst in Codespaces (Notebook 06), daarna in Codespaces (Terminal)

Je breidt de bestaande dienst `iot-status` uit. Het adres, het certificaat en je **`API-Droplet wachtwoord`** blijven hetzelfde. De API krijgt alleen leesrechten op de verzameldatabase. De ontvanger kan intussen nieuwe MQTT-metingen blijven bewaren.

**Stap 1.** Voer de cel hieronder uit. Kopieer de afgedrukte `scp`-opdracht naar een terminal in de hoofdmap van de repository. De opdracht brengt de API en een kleine service-instelling naar de thuismap van `beheer` op je server. Vul bij de terminalvraag je **`SSH beheer-wachtwoord`** in. Daarbij verschijnen geen tekens.

</div>

In [3]:
print(f"scp -o HostKeyAlgorithms=ssh-ed25519 server/webdienst.py server/configuratie/api-uitbreiding.conf beheer@{serveradres}:~/")

scp -o HostKeyAlgorithms=ssh-ed25519 server/webdienst.py server/configuratie/api-uitbreiding.conf beheer@127.0.0.1:~/


<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Na een geslaagde kopie zie je beide bestandsnamen en een voortgang die eindigt op `100%`.

**Stap 2.** Maak vanuit dezelfde Codespaces-terminal een SSH-verbinding. De volgende cel drukt de opdracht met je eigen adres af. Gebruik opnieuw je **`SSH beheer-wachtwoord`**.

</div>

In [4]:
print(f"ssh -o HostKeyAlgorithms=ssh-ed25519 beheer@{serveradres}")

ssh -o HostKeyAlgorithms=ssh-ed25519 beheer@127.0.0.1


<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** op je DigitalOcean-server (SSH-terminal)

**Stap 3.** Controleer eerst of de verzameldatabase bestaat. De opdracht toont de naam, grootte en eigenaar. Ontbreekt het bestand, hervat dan notebook 05. `sudo` kan je **`SSH beheer-wachtwoord`** vragen.

</div>

```bash
sudo ls -lh /var/lib/iot/verzamelde_metingen.db
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Stap 4.** Plaats het API-programma naast de andere serverprogramma's. De eigenaar `root` mag het programma wijzigen. De dienstgebruiker `iot` kan het lezen. Deze opdracht verandert de database en het API-wachtwoord niet.

</div>

```bash
sudo install -o root -g iot -m 0640 ~/webdienst.py /opt/iot/programmatuur/webdienst.py
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Stap 5.** Voeg de service-instelling toe. Een bestand in `iot-status.service.d` vult de bestaande dienst aan. Hier vervangt het de startopdracht door het volledige API-programma. De dienst blijft op het interne adres `127.0.0.1:8000` luisteren. Je opent geen extra firewallpoort.

</div>

```bash
sudo mkdir -p /etc/systemd/system/iot-status.service.d
sudo install -o root -g root -m 0644 ~/api-uitbreiding.conf /etc/systemd/system/iot-status.service.d/api-uitbreiding.conf
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Stap 6.** Laat systemd de gewijzigde dienstinstelling lezen en herstart alleen de webdienst. Controleer daarna de toestand.

</div>

```bash
sudo systemctl daemon-reload
sudo systemctl restart iot-status
sudo systemctl is-active iot-status
```

<div style="background-color:#e8f5e9;color:#1b5e20;border-left:5px solid #2e7d32;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;white-space:normal;"><strong>Controle</strong><br>De laatste opdracht toont <code>active</code>. Dit bewijst dat het proces draait. De volgende sectie controleert ook de HTTPS-verbinding, je aanmelding en de database-inhoud. Bij <code>failed</code> gebruik je de foutentabel onderaan dit notebook.</div>

<div style="background-color:#e3f2fd;color:#0d47a1;border-left:5px solid #1976d2;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;white-space:normal;"><strong>Tip</strong><br>Bij opnieuw doorlopen mag je dezelfde bestanden opnieuw installeren en de dienst herstarten. De database en wachtwoorden blijven behouden. De API start na een serverherstart automatisch via de bestaande dienst. Mosquitto blijft volgens notebook 03 handmatig te starten.</div>

## 4. Je eerste beveiligde aanvraag uitvoeren

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 06)

Je gebruikt de bestaande openbare `ca.crt` om de identiteit van je server te controleren. Met account `eigenaar` en je **`API-Droplet wachtwoord`** meld je je vervolgens aan. Dit is een ander wachtwoord dan dat van de MQTT-broker.

De volgende cel vraagt het **`API-Droplet wachtwoord`** in een klein invoerbalkje **helemaal bovenaan in het venster** van Codespaces, VS Code of Cursor. Vul het daar in en druk op **Enter**. Het wachtwoord wordt niet opgeslagen in het notebook, de uitvoer of Git. Het blijft alleen tijdelijk in het werkgeheugen van deze verbinding.

De aangeleverde `ApiClient` stelt certificaatcontrole, aanmelding en een wachttijd bij netwerkproblemen in. De korte pauze tussen aanvragen voorkomt dat de webdienst te veel aanvragen tegelijk krijgt. Het aanvragen zelf blijft zichtbaar in de codecellen.

</div>

In [5]:
if "api" in globals():
    api.sluiten()
api = ApiClient(f"https://{serveradres}", ca, getpass("API-Droplet wachtwoord: "))
antwoord = api.antwoord("/gezondheid")
print("HTTP-status:", antwoord.status_code)
if antwoord.status_code != 200:
    raise RuntimeError("Controleer de foutentabel voordat je verdergaat.")
print(json.dumps(antwoord.json(), ensure_ascii=False, indent=2))
assert antwoord.json().get("api_versie") == 1, "Installeer eerst de uitbreiding uit sectie 3."

HTTP-status: 200
{
  "status": "bereikbaar",
  "dienst": "practicum",
  "api_versie": 1
}


<div style="background-color:#e8f5e9;color:#1b5e20;border-left:5px solid #2e7d32;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;white-space:normal;"><strong>Controle</strong><br>Je verwacht HTTP <code>200</code>, <code>status: bereikbaar</code> en <code>api_versie: 1</code>. HTTP <code>401</code> betekent dat de server bereikbaar is maar de aanmelding niet klopt. Ontbreekt <code>api_versie</code>, dan draait nog de oorspronkelijke statusdienst.</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Vraag nu `/metadata` op. Dit antwoord beschrijft welke bronnen, eenheden en hokken beschikbaar zijn. De aantallen en inhoudsvingerafdrukken moeten overeenkomen met de controle uit notebook 05.

</div>

In [6]:
metadata = api.ophalen("/metadata")
print(json.dumps(metadata, ensure_ascii=False, indent=2))
if metadata["bronnen"] != ontvangst05["bronnen"]:
    raise ValueError("De API-inhoud verschilt van je controle uit notebook 05. Controleer of je de juiste server gebruikt en of er nieuwe data zijn toegevoegd.")
print("Beide bronnen komen overeen met notebook 05.")

{
  "api_versie": 1,
  "grens": 226677,
  "bronnen": {
    "asserva": {
      "aantal": 80028,
      "sha256": "894618c5a3275a33c2fc75841814a4d18948351c70afc8cf6c1a65eccb7f0a58"
    },
    "schauer": {
      "aantal": 146649,
      "sha256": "7d8c99caf64a6c128074b81dcf0de003d10f54887fe39a3489ebf0d52d85b249"
    }
  },
  "hokken": [
    "1",
    "10",
    "2",
    "3",
    "4",
    "5",
    "6",
    "7",
    "8",
    "9"
  ],
  "eenheden": {
    "gewichten": "kg",
    "voederopname": "g"
  },
  "datumveld": "bronrecord.DATE",
  "tijdzone": "onbevestigd; vermoedelijk Europe/Brussels",
  "selectie": "Alle ontvangen bronrecords, inclusief kwaliteitsvlaggen.",
  "paginering": "Volg volgende_na met dezelfde grens en filters. Volgorde is opslagvolgorde, niet meettijd."
}
Beide bronnen komen overeen met notebook 05.


<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Een **inhoudsvingerafdruk** is een berekende code over de recordidentificaties en hun inhoud. Gelijke aantallen alleen bewijzen niet dat je dezelfde metingen hebt. Daarom vergelijk je hier zowel de aantallen als deze codes. Dit is nog geen controle of iedere meting geschikt is voor een groeiberekening.

</div>

## 5. De API-documentatie en een meetrecord lezen

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 06)

Een API heeft documentatie nodig, zodat de client weet welke aanvragen mogelijk zijn. FastAPI beschrijft de beschikbare paden en parameters in het gestandaardiseerde **OpenAPI-formaat**. Je vraagt die documentatie op via dezelfde beveiligde verbinding. De tabel hieronder wordt uit dat antwoord opgebouwd.

</div>

In [7]:
documentatie = api.ophalen("/openapi.json")
parameterschema = documentatie["paths"]["/metingen"]["get"]["parameters"]
display(pd.DataFrame([{"parameter": p["name"], "verplicht": p.get("required", False),
                       "schema": json.dumps(p["schema"], ensure_ascii=False)} for p in parameterschema]))

,parameter,verplicht,schema
0,meettype,False,"{""anyOf"": [{""enum"": [""gewichten"", ""voederopnam..."
1,hok,False,"{""anyOf"": [{""type"": ""string"", ""maxLength"": 100..."
2,dier_id,False,"{""anyOf"": [{""type"": ""string"", ""maxLength"": 100..."
3,vanaf,False,"{""anyOf"": [{""type"": ""string"", ""format"": ""date""..."
4,tot,False,"{""anyOf"": [{""type"": ""string"", ""format"": ""date""..."
5,limiet,False,"{""type"": ""integer"", ""maximum"": 1000, ""minimum""..."
6,na,False,"{""type"": ""integer"", ""maximum"": 922337203685477..."
7,grens,False,"{""anyOf"": [{""type"": ""integer"", ""maximum"": 9223..."


| Pad | Gebruik |
|---|---|
| `/gezondheid` | Bereikbaarheid en API-versie |
| `/metadata` | Bronnen, aantallen, hokken, eenheden en inhoudscontrole |
| `/metingen` | Meetrecords, eventueel gefilterd |
| `/openapi.json` | Beschrijving van de beschikbare API-aanvragen |

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Vraag één record op van je eigen meettype. De buitenste JSON bevat informatie over het antwoord. De lijst `metingen` bevat de eigenlijke records. Een lijst kan nul, één of meerdere elementen bevatten. Met `[0]` selecteer je het eerste element wanneer de lijst niet leeg is.

</div>

In [8]:
eerste_pagina = api.ophalen("/metingen", {"meettype": rol, "limiet": 1})
print(json.dumps(eerste_pagina, ensure_ascii=False, indent=2))
if not eerste_pagina["metingen"]:
    raise ValueError("Je eigen meettype bevat geen records. Controleer notebook 05.")
eerste = eerste_pagina["metingen"][0]
print("Record:", eerste["record_id"])
print("Meetwaarde:", eerste["waarde"], eerste["eenheid"])
print("Hokmetadata:", eerste["hokmetadata"])

{
  "api_versie": 1,
  "grens": 226677,
  "aantal": 1,
  "volgende_na": 1,
  "metingen": [
    {
      "bron": "asserva",
      "bronrecord": {
        "DATAORIGIN": "allocation",
        "DATE": "2023-08-24",
        "DTMARKER": 0.5,
        "DTMARKER_INT": 0.0,
        "EID": "89728010",
        "ERROR": "no error",
        "FILTER": "0",
        "FLAG_VBW_hampel": "0",
        "FLAG_VBW_matern1": "0",
        "FLAG_VBW_matern2": "0",
        "FLAG_VBW_polyn": "0",
        "GENDER": "0",
        "PEN": "3",
        "PROCESSTIME": 0.0,
        "ROW_ID": "897280101692878400000000000",
        "TIMESTAMPstart": "20230824120000",
        "TIMESTAMPstop": "20230824120001",
        "TIMEstart": "12:00:00",
        "TIMEstop": "12:00:01",
        "TREAT": "C",
        "TRIALDAY": 0.5,
        "VBW": 6.68
      },
      "dier_id": "89728010",
      "eenheid": "kg",
      "hok": "3",
      "hokmetadata": {
        "afdeling": 1,
        "gebouw": "M5",
        "hok": "3",
        "oppervlakte

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Vraag 2.** Welke eenheid heeft jouw eerste record, en waar vind je de hokoppervlakte terug?

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Jouw antwoord (Vraag 2):**

*Voeg hier je antwoord in. Tip: gebruik de velden eenheid en hokmetadata in jouw uitvoer.*

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Antwoord 2**

Een gewichtsrecord bevat kg en een voederopnamerecord g. De hokoppervlakte staat in hokmetadata, in het veld oppervlakte_m2. Voor deze oefening is dat 8. De API bewaart dezelfde broninhoud en hokmetadata die via MQTT zijn ontvangen.

</div>

## 6. Zelf een gerichte selectie opvragen

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 06)

Een selectie maakt het mogelijk om bijvoorbeeld één hok gedurende enkele dagen te onderzoeken. De API ondersteunt `meettype`, `hok`, `dier_id`, `vanaf` en `tot`. Datums worden vergeleken met het bronveld `DATE`. `vanaf` hoort bij de selectie, `tot` niet. De periode vanaf `2023-08-24` tot `2023-08-27` omvat dus 24, 25 en 26 augustus.

**Doe-opdracht.** Kies een hok uit de metadata. Vraag vijf voederopnamerecords op voor de bovenstaande periode, ook als jouw oorspronkelijke rol gewichten is. Gebruik de voorbeeldsleutels uit sectie 2. Een hokcode is tekst, bijvoorbeeld `"1"`.

Vul in de volgende cel het woordenboek `selectie` aan. Vraag daarna de selectie op met `api.ophalen`. De aangeleverde controlecel toont de belangrijkste velden.

</div>

In [9]:
# SCHRIJF HIER JE CODE:
# 1. Maak selectie met meettype, hok, vanaf, tot en limiet.
# 2. Kies een hokcode uit metadata["hokken"].
# 3. Bewaar api.ophalen("/metingen", selectie) als geselecteerd.

In [10]:
selectie = {"meettype": "voederopname", "hok": metadata["hokken"][0],
            "vanaf": "2023-08-24", "tot": "2023-08-27", "limiet": 5}
geselecteerd = api.ophalen("/metingen", selectie)

In [11]:
assert selectie["meettype"] == "voederopname" and selectie["limiet"] == 5
assert selectie["hok"] in metadata["hokken"]
assert selectie["vanaf"] == "2023-08-24" and selectie["tot"] == "2023-08-27"
assert geselecteerd["metingen"], "Deze selectie is leeg. Kies een ander hok."
for b in geselecteerd["metingen"]:
    assert b["hok"] == selectie["hok"] and b["meettype"] == selectie["meettype"]
    assert selectie["vanaf"] <= b["bronrecord"]["DATE"] < selectie["tot"]
display(pd.DataFrame([{"record_id": b["record_id"], "hok": b["hok"],
                       "datum": b["bronrecord"]["DATE"], "voederopname_g": b["waarde"]}
                      for b in geselecteerd["metingen"]]))

,record_id,hok,datum,voederopname_g
0,897280261692879735000000000,1,2023-08-24,0.0
1,897280261692888293000000000,1,2023-08-24,1.0
2,897280261692888311000000000,1,2023-08-24,0.0
3,897280261692895329000000000,1,2023-08-24,0.0
4,897280261692897467000000000,1,2023-08-24,0.0


<div style="background-color:#e8f5e9;color:#1b5e20;border-left:5px solid #2e7d32;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;white-space:normal;"><strong>Interpretatie</strong><br>De vijf records zijn een beperkte pagina, geen dagsom of gemiddelde. De volgorde is de opslagvolgorde op de server. Een klein antwoord is handig om de structuur te leren kennen. Voor een volledige analyse moet je alle pagina’s ophalen.</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Vraag 3.** Waarom kun je met deze vijf records nog niet de gemiddelde voederopname per dag voor het hok berekenen?

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Jouw antwoord (Vraag 3):**

*Voeg hier je antwoord in. Tip: denk aan limiet, individuele bezoeken en het aantal dieren.*

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Antwoord 3**

De limiet beperkt het antwoord tot vijf bezoekrecords. Andere bezoeken van die dagen ontbreken. Bovendien moet je bepalen welke dieridentificaties geldig zijn en hoe je per dag en per dier of hok aggregeert. Een gemiddelde van vijf willekeurige bezoekwaarden is geen gemiddelde dagelijkse voederopname.

</div>

## 7. Een lege selectie en een fout onderscheiden

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 06)

Een goed geformuleerde vraag kan nul resultaten opleveren. Een ongeldige aanvraag heeft een andere betekenis. Je vergelijkt beide situaties bewust. Vraag eerst een hok op dat niet in de dataset voorkomt. Vraag daarna een pagina met `limiet=0` op, terwijl de documentatie minstens één record als paginalimiet voorschrijft.

</div>

In [12]:
leeg = api.antwoord("/metingen", {"hok": "bestaat-niet"})
fout = api.antwoord("/metingen", {"limiet": 0})
print("Lege selectie:", leeg.status_code, leeg.json())
print("Ongeldige aanvraag:", fout.status_code)
print(json.dumps(fout.json(), ensure_ascii=False, indent=2))
assert leeg.status_code == 200 and leeg.json()["metingen"] == []
assert fout.status_code == 422

Lege selectie: 200 {'api_versie': 1, 'grens': 226677, 'aantal': 0, 'volgende_na': None, 'metingen': []}
Ongeldige aanvraag: 422
{
  "detail": [
    {
      "type": "greater_than_equal",
      "loc": [
        "query",
        "limiet"
      ],
      "msg": "Input should be greater than or equal to 1",
      "input": "0",
      "ctx": {
        "ge": 1
      }
    }
  ]
}


<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Vraag 4.** Welke handeling past bij HTTP 200 met een lege lijst, en welke bij HTTP 422?

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Jouw antwoord (Vraag 4):**

*Voeg hier je antwoord in. Tip: vergelijk een ontbrekende selectie met een verkeerde parameter.*

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Antwoord 4**

Bij 200 is de aanvraag geldig. Ik controleer of er voor mijn hok en periode gegevens beschikbaar zijn. Bij 422 herstel ik de ongeldige parameter, hier limiet=0, en stuur ik de aanvraag opnieuw. Geen van beide vraagt om het opnieuw installeren van de server.

</div>

## 8. Beide bronnen in een analysedatabase bewaren

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 06)

De eerste vier weken bevatten samen 226.677 bronrecords. Je haalt ze op in pagina's van maximaal duizend records. Het veld `volgende_na` geeft aan waar de volgende pagina begint. De bijbehorende `grens` houdt de bovenkant van deze overdracht vast. Nieuwe records die tijdens het ophalen op de server aankomen, komen bij de volgende volledige ophaalronde mee.

De aangeleverde functie `werk_analyse_bij` volgt alle pagina's, controleert de inhoud en bewaart de records in **`gegevens/werk/analyse_06.db`**. Bij de eerste uitvoering maakt ze dit lege analysebestand aan. Bij een volgende uitvoering behoudt ze bestaande metingen en voegt ze alleen nieuwe toe. Bij een fout draait ze de wijzigingen van die ophaalronde terug.

Je eigen `asserva.db` of `schauer.db` blijft de brondatabase voor publicatie. Geef dat bestand nooit als analysedatabase op. Onderstaande controle laat zien of je begint met een nieuw of bestaand analysebestand.

</div>

In [13]:
analysepad = werkmap / "analyse_06.db"
print("Analysebestand:", analysepad.relative_to(hoofdmap))
print("Bestaand analysebestand wordt bijgewerkt." if analysepad.exists() else "Er wordt een nieuwe, lege analysedatabase aangemaakt.")
eerste_import = werk_analyse_bij(api.ophalen, analysepad)
if eerste_import["bronnen"] != ontvangst05["bronnen"]:
    raise ValueError("Deze gegevens horen niet meer bij de gecontroleerde beginsituatie uit notebook 05.")
print(json.dumps(eerste_import, ensure_ascii=False, indent=2))

Analysebestand: gegevens/werk/analyse_06.db
Er wordt een nieuwe, lege analysedatabase aangemaakt.


25,000 records gecontroleerd


50,000 records gecontroleerd


75,000 records gecontroleerd


100,000 records gecontroleerd


125,000 records gecontroleerd


150,000 records gecontroleerd


175,000 records gecontroleerd


200,000 records gecontroleerd


225,000 records gecontroleerd


Klaar: 226,677 nieuwe records, 0 ongewijzigde records.
{
  "gecontroleerd_op_utc": "2026-11-25T09:05:20.456592+00:00",
  "nieuw": 226677,
  "ongewijzigd": 0,
  "bronnen": {
    "asserva": {
      "aantal": 80028,
      "sha256": "894618c5a3275a33c2fc75841814a4d18948351c70afc8cf6c1a65eccb7f0a58"
    },
    "schauer": {
      "aantal": 146649,
      "sha256": "7d8c99caf64a6c128074b81dcf0de003d10f54887fe39a3489ebf0d52d85b249"
    }
  },
  "grens": 226677
}


<div style="background-color:#e8f5e9;color:#1b5e20;border-left:5px solid #2e7d32;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;white-space:normal;"><strong>Controle</strong><br>De eerste import in een nieuw bestand voegt 80.028 gewichtsrecords en 146.649 voederopnamerecords toe. Een bestaande correcte analysedatabase kan nul nieuwe records melden. De aantallen en inhoudsvingerafdrukken moeten in beide gevallen overeenkomen met notebook 05. De voortgang verschijnt per 25 pagina’s.</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Bekijk nu met SQL wat er lokaal is opgeslagen. Deze query leest de analysedatabase in Codespaces. Ze stuurt geen nieuwe aanvraag naar de API. De eerste en laatste bronkalenderdatum helpen je controleren welke periode beschikbaar is.

</div>

In [14]:
with closing(sqlite3.connect(analysepad.as_uri() + "?mode=ro", uri=True)) as db:
    samenvatting = pd.read_sql_query("""
        SELECT bron, meettype, eenheid, COUNT(*) AS records,
               COUNT(DISTINCT hok) AS hokken, MIN(datum) AS eerste_dag, MAX(datum) AS laatste_dag
        FROM metingen GROUP BY bron, meettype, eenheid ORDER BY bron
    """, db)
display(samenvatting)
assert set(samenvatting["bron"]) == {"asserva", "schauer"}

,bron,meettype,eenheid,records,hokken,eerste_dag,laatste_dag
0,asserva,gewichten,kg,80028,10,2023-08-24,2023-09-20
1,schauer,voederopname,g,146649,10,2023-08-24,2023-09-20


<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

De analysedatabase bevat ook de volledige JSON-inhoud en de kwaliteitsvlaggen van ieder bronrecord. De API voert geen selectie van geldige dieren of groeimetingen uit. Die analysekeuzes volgen bij de berekening. De aantallen hier zijn dus aantallen bronrecords, geen aantallen bruikbare bezoeken.

</div>

## 9. Herhaald ophalen en verderwerken testen

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 06)

Bij dagelijks gebruik vraag je gegevens vaker dan één keer op. De combinatie van bron en recordidentificatie bepaalt of een meting al aanwezig is. Voer de volledige import opnieuw uit. Dit vraagt opnieuw alle gegevens op om hun inhoud te controleren. Zonder nieuwe publicaties verwacht je nul nieuwe records.

</div>

In [15]:
herhaalde_import = werk_analyse_bij(api.ophalen, analysepad)
assert herhaalde_import["nieuw"] == 0, "Er zijn nieuwe bronrecords aangekomen. Controleer dit voordat je de herhaalproef beoordeelt."
assert herhaalde_import["bronnen"] == eerste_import["bronnen"]
print("Herhaling voegt geen dubbele records toe.")

25,000 records gecontroleerd


50,000 records gecontroleerd


75,000 records gecontroleerd


100,000 records gecontroleerd


125,000 records gecontroleerd


150,000 records gecontroleerd


175,000 records gecontroleerd


200,000 records gecontroleerd


225,000 records gecontroleerd


Klaar: 0 nieuwe records, 226,677 ongewijzigde records.
Herhaling voegt geen dubbele records toe.


<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Vraag 5.** Wat gebeurt er met je lokale analysedatabase als je Codespace stopt, of als de API even niet bereikbaar is?

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Jouw antwoord (Vraag 5):**

*Voeg hier je antwoord in. Tip: onderscheid bestaande bestanden, nieuwe aanvragen en een volledig nieuwe Codespace.*

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Antwoord 5**

Het opgeslagen analysebestand blijft bestaan bij stoppen en hervatten van dezelfde Codespace. Ik kan de aanwezige gegevens analyseren zonder nieuwe API-aanvraag. Voor een nieuwe ophaalronde moet de API bereikbaar zijn. Een volledig nieuwe Codespace bevat dit genegeerde werkbestand niet. Ik kan het dan opnieuw via de API opbouwen zolang de servergegevens beschikbaar zijn.

</div>

<div style="background-color:#e3f2fd;color:#0d47a1;border-left:5px solid #1976d2;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;white-space:normal;"><strong>Tip</strong><br>Hervat later dezelfde Codespace en voer de cellen opnieuw uit. Het wachtwoord wordt opnieuw gevraagd, maar het analysebestand blijft behouden. Bij een onderbroken import herhaal je de volledige importcel. Je hoeft daarvoor geen databases te verwijderen.</div>

## 10. Resultaat bewaren en afsluiten

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 06)

Het controlebestand wordt automatisch opgebouwd uit je uitgevoerde aanvragen, selectie, foutproef en importcontroles. Je schrijft dit verslag niet zelf. Je eigen uitleg bij de vragen blijft in het notebook staan. Het controlebestand bevat geen wachtwoord of serveradres.

</div>

In [16]:
assert leeg.status_code == 200 and fout.status_code == 422
assert herhaalde_import["nieuw"] == 0
assert eerste_import["bronnen"] == herhaalde_import["bronnen"] == ontvangst05["bronnen"]
verslag = {
    "deelnemerscode": deelnemer, "branch": branch, "rol": rol,
    "getest_op_utc": datetime.now(timezone.utc).isoformat(), "api_versie": metadata["api_versie"],
    "https_en_aanmelding": "geslaagd", "selectie": selectie,
    "selectie_aantal": geselecteerd["aantal"], "lege_selectie_status": leeg.status_code,
    "ongeldige_aanvraag_status": fout.status_code,
    "eerste_import": eerste_import, "herhaalde_import": herhaalde_import,
    "inhoud_gelijk_aan_05": True, "analysebestand": "gegevens/werk/analyse_06.db",
}
inleverpad = hoofdmap / "gegevens/inlevering/06_apicontrole.json"
inleverpad.write_text(json.dumps(verslag, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
api.sluiten()
print("Bewaard:", inleverpad.relative_to(hoofdmap))
print("De API-sessie is gesloten. Voor nieuwe aanvragen voer je de aanmeldcel opnieuw uit.")

Bewaard: gegevens/inlevering/06_apicontrole.json
De API-sessie is gesloten. Voor nieuwe aanvragen voer je de aanmeldcel opnieuw uit.


<div style="background-color:#fff8e1;color:#6d4c00;border-left:5px solid #f9a825;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;white-space:normal;"><strong>Stop je Codespace na het practicum</strong><br>Bewaar je werk, commit en push voordat je de Codespace stopt. De Droplet heeft een afzonderlijke facturering. Laat die beschikbaar voor de volgende oefening en verwijder hem alleen volgens de afspraak met de docent.</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Source Control) en op de GitHub-website

1. Vul je antwoorden aan en kies **File → Save All**.
2. Selecteer `notebooks/06_gegevens_opvragen_via_een_api.ipynb` en `gegevens/inlevering/06_apicontrole.json` in **Source Control**. Controleer de uitvoer en laat tijdelijke regels met je serveradres weg als je die niet wilt publiceren.
3. Commit bijvoorbeeld met **API-aanvragen en analysedatabase gecontroleerd**. Kies **Push** en controleer beide bestanden op jouw GitHub-branch.
4. `analyse_06.db` blijft onder `gegevens/werk/` en wordt niet gecommit. Bij het verwijderen van je Codespace kun je dit bestand eerst downloaden nadat de importcel klaar is. De code sluit de SQLite-verbinding na iedere import.
5. Sluit de SSH-terminal met `exit`. Stop de Codespace via [github.com/codespaces](https://github.com/codespaces). Alleen het browsertabblad sluiten is onvoldoende.

In notebook 07 gebruik je de analysedatabase om groei, voederopname en voederconversie per hok te onderzoeken. Bewaar daarom dezelfde Codespace en het analysebestand.

</div>

## Hulp bij veelvoorkomende problemen

| Probleem | Controle en herstel |
|---|---|
| Controlebestand 05 ontbreekt | Rond notebook 05 en de automatische inlevercel af. Maak geen fictief controlebestand. |
| ImportError voor api_ophalen of webdienst | Controleer of de bestanden van notebook 06 op jouw branch staan en correct zijn gekopieerd. |
| Certificaatcontrole mislukt | Controleer serveradres, geldigheid en het via SSH opgehaalde ca.crt. Schakel de controle niet uit. |
| HTTP 401 | Voer de aanmeldcel opnieuw uit met je **API-Droplet wachtwoord**. |
| HTTP 404 voor /metingen | Controleer de installatie van de uitbreiding en herstart iot-status. |
| HTTP 422 | Lees de ongeldige parameter in detail. Gebruik geldige datums en een limiet van 1 tot 1000. |
| HTTP 429 | Er kwamen te veel aanvragen. Wacht even en gebruik de aangeleverde ApiClient. |
| HTTP 502 | Nginx kan de interne Python-dienst niet bereiken. Controleer iot-status. |
| HTTP 503 | De verzameldatabase ontbreekt of is niet leesbaar. Controleer notebook 05, het bestand en de dienstrechten. |
| HTTP 409 of inhoudsverschil | De serverdatabase is vervangen of gewijzigd. Controleer de server en de broncontroles. Wis geen bronbestanden om de fout te omzeilen. |
| Lege bron bij import | Ontvang eerst beide volledige beginsets via notebook 05. |
| Geen analysedatabase van notebook 06 | Gebruik het aparte bestand analyse_06.db. Geef geen bron- of verzameldatabase op. |
| Onderbroken import | Herstel de verbinding en herhaal de importcel. Wijzigingen van de mislukte ronde zijn teruggedraaid. |
| NameError na hervatten | Voer de codecellen opnieuw in volgorde uit. Bestanden blijven bestaan, Python-variabelen niet. |

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** op je DigitalOcean-server (SSH-terminal)

Deze opdrachten tonen de dienstinstelling en recente logregels. Ze drukken geen opgeslagen wachtwoordbestand af.

</div>

```bash
sudo systemctl status iot-status --no-pager
sudo journalctl -u iot-status -n 30 --no-pager
sudo systemctl cat iot-status
```

<div style="background-color:#e3f2fd;color:#0d47a1;border-left:5px solid #1976d2;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;white-space:normal;"><strong>Tip</strong><br>Als de nieuwe API niet start, kun je tijdelijk terug naar de bereikbaarheidstest uit notebook 03. Hernoem alleen de aanvullende instelling en herstart de dienst met onderstaande opdrachten. De verzameldatabase blijft behouden. Herhaal daarna de installatie van deze uitbreiding zodra de oorzaak is opgelost.</div>

```bash
sudo mv /etc/systemd/system/iot-status.service.d/api-uitbreiding.conf /etc/systemd/system/iot-status.service.d/api-uitbreiding.conf.uit
sudo systemctl daemon-reload
sudo systemctl restart iot-status
```

## Technische bronnen

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

- [FastAPI: queryparameters en validatie](https://fastapi.tiangolo.com/tutorial/query-params-str-validations/)
- [FastAPI: HTTP Basic-aanmelding](https://fastapi.tiangolo.com/advanced/security/http-basic-auth/)
- [Requests: sessies en certificaatcontrole](https://requests.readthedocs.io/en/latest/user/advanced/)
- [SQLite: database uitsluitend lezen met mode=ro](https://www.sqlite.org/uri.html)
- [Gegevenswoordenboek](../../gegevens/gegevenswoordenboek.md)
- [API-gegevenscontract](../../documentatie/api_gegevenscontract.md)

De technische documentatie is gecontroleerd op 25 november 2026.

</div>